# Day 32 — DL Inference End-to-End Fix

Fixes the Day 31 DL button by wiring the saved Day 12 LSTM/GRU artifacts to `/api/predict/dl/`, enforcing exact Day 11 feature order, and applying saved imputer/scaler. It does not retrain models or invent scores. Run from the IPL repository with the project Conda environment selected. Backups are made before URL changes.

In [6]:

from pathlib import Path
import json, re, shutil, datetime, subprocess, sys
import pandas as pd

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "django_api" / "manage.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from the IPL repository or a subfolder.")
DJANGO_DIR = ROOT / "django_api"
APP_DIR = DJANGO_DIR / "ipl_api"
URLS = APP_DIR / "urls.py"
manifest_candidates = [ROOT/"reports"/"day12"/"model_artifacts_manifest.csv", ROOT/"reports"/"day12_model_artifacts_manifest.csv"]
manifest_candidates += list((ROOT/"reports").rglob("*model*artifact*manifest*.csv")) if (ROOT/"reports").exists() else []
MANIFEST = next((p for p in manifest_candidates if p.is_file()), None)
if MANIFEST is None:
    raise FileNotFoundError("Day 12 model_artifacts_manifest.csv not found under reports/. Run Day 12 first.")
manifest = pd.read_csv(MANIFEST)
print("Repository:", ROOT)
print("Manifest:", MANIFEST.relative_to(ROOT))
print("Manifest columns:", list(manifest.columns))
display(manifest)


Repository: D:\Git\Personal\ipl-player-performance
Manifest: reports\day12\model_artifacts_manifest.csv
Manifest columns: ['dataset', 'target', 'selected_architecture', 'model_path', 'imputer_path', 'scaler_path', 'feature_manifest_path']


,dataset,target,selected_architecture,model_path,imputer_path,scaler_path,feature_manifest_path
0,batting,next_match_runs,LSTM,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...
1,bowling,next_match_wickets,GRU,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...
2,bowling,next_match_runs_conceded,GRU,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...,D:\Git\Personal\ipl-player-performance\models\...


## Inspect saved artifact manifest and feature order

The endpoint will only work if Day 12 successfully saved a model, imputer/scaler, and Day 11 feature manifest for the selected target.

In [7]:
def resolve_path(value):
    p = Path(str(value))
    candidates = [p] if p.is_absolute() else [ROOT/p, ROOT/"models"/p, MANIFEST.parent/p]
    parts = list(p.parts)
    if "models" in parts: candidates.append(ROOT.joinpath(*parts[parts.index("models"):]))
    return next((c.resolve() for c in candidates if c.exists()), candidates[0].resolve())
for col in ["model_path","imputer_path","scaler_path","feature_manifest_path"]:
    if col in manifest.columns:
        print("\\n"+col)
        for v in manifest[col].dropna().astype(str).unique():
            print(v, "=>", resolve_path(v), "exists:", resolve_path(v).exists())
for _, row in manifest.iterrows():
    fpcol = next((c for c in ["feature_manifest_path","features_path","feature_path"] if c in manifest.columns and pd.notna(row.get(c))), None)
    if fpcol:
        fp = resolve_path(row[fpcol])
        if fp.exists():
            info = json.loads(fp.read_text(encoding="utf-8"))
            features = info.get("feature_columns_in_order", info.get("feature_columns", []))
            print("\\nTarget", row.get("target", row.get("target_name")), "| features:", len(features), features)


\nmodel_path
D:\Git\Personal\ipl-player-performance\models\day12\batting_next_match_runs_lstm.keras => D:\Git\Personal\ipl-player-performance\models\day12\batting_next_match_runs_lstm.keras exists: True
D:\Git\Personal\ipl-player-performance\models\day12\bowling_next_match_wickets_gru.keras => D:\Git\Personal\ipl-player-performance\models\day12\bowling_next_match_wickets_gru.keras exists: True
D:\Git\Personal\ipl-player-performance\models\day12\bowling_next_match_runs_conceded_gru.keras => D:\Git\Personal\ipl-player-performance\models\day12\bowling_next_match_runs_conceded_gru.keras exists: True
\nimputer_path
D:\Git\Personal\ipl-player-performance\models\day12\batting_next_match_runs_lstm_imputer.joblib => D:\Git\Personal\ipl-player-performance\models\day12\batting_next_match_runs_lstm_imputer.joblib exists: True
D:\Git\Personal\ipl-player-performance\models\day12\bowling_next_match_wickets_gru_imputer.joblib => D:\Git\Personal\ipl-player-performance\models\day12\bowling_next_match_wi

## Write the inference view and register URL

In [8]:
ENDPOINT_SOURCE = 'import json\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom django.conf import settings\nfrom django.http import JsonResponse\nfrom django.views.decorators.csrf import csrf_exempt\nfrom django.views.decorators.http import require_POST\n\nTARGETS = {"next_match_runs", "next_match_wickets", "next_match_runs_conceded"}\n\ndef repo_root():\n    base = Path(settings.BASE_DIR).resolve()\n    return base.parent if base.name == "django_api" else base\n\ndef resolve_artifact(root, value, base_dir=None):\n    p = Path(str(value))\n    candidates = [p] if p.is_absolute() else [root/p, root/"models"/p]\n    if base_dir and not p.is_absolute():\n        candidates.append(Path(base_dir)/p)\n    parts = list(p.parts)\n    if "models" in parts:\n        candidates.append(root.joinpath(*parts[parts.index("models"):]))\n    for c in candidates:\n        if c.exists():\n            return c.resolve()\n    return candidates[0].resolve()\n\ndef value_from(row, names):\n    for name in names:\n        value = row.get(name)\n        if value is not None and not pd.isna(value) and str(value).strip():\n            return str(value).strip()\n    return None\n\ndef find_manifest(root):\n    candidates = [root/"reports"/"day12"/"model_artifacts_manifest.csv",\n                  root/"reports"/"day12_model_artifacts_manifest.csv"]\n    if (root/"reports").exists():\n        candidates += list((root/"reports").rglob("*model*artifact*manifest*.csv"))\n    path = next((p for p in candidates if p.is_file()), None)\n    if path is None:\n        raise FileNotFoundError("Day 12 model_artifacts_manifest.csv not found under reports/.")\n    return pd.read_csv(path), path\n\n@csrf_exempt\n@require_POST\ndef deep_learning_prediction(request):\n    try:\n        payload = json.loads(request.body.decode("utf-8"))\n    except Exception:\n        return JsonResponse({"detail": "Request body must be valid JSON."}, status=400)\n    target = str(payload.get("target", "")).strip()\n    if target not in TARGETS:\n        return JsonResponse({"detail": "Unsupported target.", "allowed_targets": sorted(TARGETS)}, status=400)\n    if payload.get("preprocessed") is True:\n        return JsonResponse({"detail": "Send raw values in Day 11 feature order. The endpoint applies saved Day 12 preprocessing; remove preprocessed:true."}, status=400)\n    try:\n        sequence = np.asarray(payload.get("sequence"), dtype=np.float32)\n    except (TypeError, ValueError):\n        return JsonResponse({"detail": "sequence must contain numeric values."}, status=400)\n    if sequence.ndim != 2 or sequence.shape[0] != 5:\n        return JsonResponse({"detail": "sequence must be exactly 5 chronological rows.", "received_shape": list(sequence.shape)}, status=400)\n    if not np.isfinite(sequence).all():\n        return JsonResponse({"detail": "sequence contains NaN or infinite values."}, status=400)\n\n    root = repo_root()\n    try:\n        manifest, manifest_path = find_manifest(root)\n    except Exception as exc:\n        return JsonResponse({"detail": str(exc)}, status=503)\n    target_col = next((c for c in ["target", "target_name", "prediction_target"] if c in manifest.columns), None)\n    if not target_col:\n        return JsonResponse({"detail": "Manifest is missing a target column.", "columns": list(manifest.columns)}, status=503)\n    rows = manifest[manifest[target_col].astype(str).str.strip().eq(target)]\n    if rows.empty:\n        return JsonResponse({"detail": "No saved DL model is recorded for this target.", "target": target}, status=503)\n    row = rows.iloc[0].to_dict()\n    model_value = value_from(row, ["model_path", "model_file", "artifact_path"])\n    feature_value = value_from(row, ["feature_manifest_path", "features_path", "feature_path"])\n    if not model_value or not feature_value:\n        return JsonResponse({"detail": "Manifest row must include model_path and feature_manifest_path.", "columns": list(manifest.columns)}, status=503)\n    model_path = resolve_artifact(root, model_value, manifest_path.parent)\n    feature_path = resolve_artifact(root, feature_value, manifest_path.parent)\n    if not model_path.is_file():\n        return JsonResponse({"detail": "Saved DL model not found.", "path": str(model_path)}, status=503)\n    if not feature_path.is_file():\n        return JsonResponse({"detail": "Day 11 feature manifest not found.", "path": str(feature_path)}, status=503)\n    try:\n        feature_info = json.loads(feature_path.read_text(encoding="utf-8"))\n        features = feature_info.get("feature_columns_in_order", feature_info.get("feature_columns"))\n        if not isinstance(features, list) or not features:\n            raise ValueError("feature_columns_in_order missing or empty")\n    except Exception as exc:\n        return JsonResponse({"detail": "Cannot read ordered feature manifest.", "path": str(feature_path), "error": str(exc)}, status=503)\n    if sequence.shape[1] != len(features):\n        return JsonResponse({"detail": "Feature count mismatch.", "expected_feature_count": len(features),\n                             "received_feature_count": int(sequence.shape[1]), "feature_names_in_order": features,\n                             "hint": "Provide five rows with one number per feature in this exact order."}, status=400)\n\n    imputer_value = value_from(row, ["imputer_path", "imputer_file"])\n    scaler_value = value_from(row, ["scaler_path", "scaler_file"])\n    try:\n        flat = sequence.reshape(5, len(features))\n        if imputer_value:\n            path = resolve_artifact(root, imputer_value, manifest_path.parent)\n            if not path.is_file():\n                return JsonResponse({"detail": "Saved imputer not found.", "path": str(path)}, status=503)\n            flat = joblib.load(path).transform(flat)\n        if scaler_value:\n            path = resolve_artifact(root, scaler_value, manifest_path.parent)\n            if not path.is_file():\n                return JsonResponse({"detail": "Saved scaler not found.", "path": str(path)}, status=503)\n            flat = joblib.load(path).transform(flat)\n        x = np.asarray(flat, dtype=np.float32).reshape(1, 5, len(features))\n        import tensorflow as tf\n        model = tf.keras.models.load_model(model_path)\n        expected = tuple(model.input_shape[1:])\n        if len(expected) != 2 or expected[0] not in (None, 5) or expected[1] not in (None, len(features)):\n            return JsonResponse({"detail": "Model input shape does not match feature manifest.",\n                                 "model_input_shape": list(model.input_shape), "feature_count": len(features)}, status=503)\n        prediction = float(np.asarray(model.predict(x, verbose=0)).reshape(-1)[0])\n    except Exception as exc:\n        return JsonResponse({"detail": "DL inference failed while loading or applying saved artifacts.",\n                             "error": str(exc), "model_path": str(model_path)}, status=503)\n    return JsonResponse({"target": target, "prediction": prediction,\n                         "model": value_from(row, ["architecture", "model_type", "selected_architecture"]) or "Saved LSTM/GRU",\n                         "sequence_length": 5, "feature_count": len(features), "feature_names_in_order": features,\n                         "preprocessing": {"imputer_applied": bool(imputer_value), "scaler_applied": bool(scaler_value)}})\n'

import datetime, shutil, re
backup = URLS.with_name(URLS.name + ".day32_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S") + ".bak")
shutil.copy2(URLS, backup)
text = URLS.read_text(encoding="utf-8")
imp = "from .deep_learning_prediction_day32 import deep_learning_prediction"
if imp not in text:
    lines = text.splitlines()
    last = max((i for i,line in enumerate(lines) if line.startswith("import ") or line.startswith("from ")), default=-1)
    lines.insert(last+1, imp)
    text = "\n".join(lines) + "\n"
if 'name="predict-dl-day32"' not in text:
    m = re.search(r"(?s)(urlpatterns\s*=\s*\[)(.*?)(\])", text)
    if not m:
        raise RuntimeError("Could not find urlpatterns list; original URLs were backed up.")
    body = m.group(2)
    body += '\n    path("predict/dl/", deep_learning_prediction, name="predict-dl-day32"),\n'
    text = text[:m.start()] + m.group(1) + body + m.group(3) + text[m.end():]
URLS.write_text(text, encoding="utf-8")
endpoint_path = APP_DIR / "deep_learning_prediction_day32.py"
endpoint_path.write_text(ENDPOINT_SOURCE, encoding="utf-8")
print("Backup:", backup.name)
print("Endpoint:", endpoint_path.relative_to(ROOT))
print("Registered endpoint: /api/predict/dl/")


Backup: urls.py.day32_20261024_020154.bak
Endpoint: django_api\ipl_api\deep_learning_prediction_day32.py
Registered endpoint: /api/predict/dl/


## Update dashboard DL button and response handler

In [9]:
DL_JS_SOURCE = '(() => {\n  document.addEventListener("click", async event => {\n    const button = event.target.closest("[data-day31-dl-submit]");\n    if (!button) return;\n    const panel = button.closest(".card") || button.closest(".col") || button.parentElement?.parentElement || document;\n    const area = panel.querySelector("textarea");\n    const out = panel.querySelector("pre, .prediction-output, .result, .output") || panel.lastElementChild;\n    const show = value => { if (out) out.textContent = typeof value === "string" ? value : JSON.stringify(value, null, 2); };\n    if (!area) return show("DL sequence input not found.");\n    let payload;\n    try { payload = JSON.parse(area.value); } catch (_) { return show("Enter valid JSON."); }\n    if (!["next_match_runs","next_match_wickets","next_match_runs_conceded"].includes(payload.target))\n      return show({detail:"Use a supported target: next_match_runs, next_match_wickets, next_match_runs_conceded."});\n    if (!Array.isArray(payload.sequence) || payload.sequence.length !== 5 ||\n        !payload.sequence.every(r => Array.isArray(r) && r.length > 0 && r.every(v => typeof v === "number" && Number.isFinite(v))))\n      return show({detail:"sequence must contain five chronological arrays of finite numeric feature values."});\n    if (payload.preprocessed === true) return show({detail:"Remove preprocessed:true. Send raw Day 11 ordered features; server applies Day 12 imputer/scaler."});\n    button.disabled = true; show("Running saved DL model…");\n    try {\n      const response = await fetch("/api/predict/dl/", {method:"POST", credentials:"same-origin",\n        headers:{"Content-Type":"application/json","Accept":"application/json"}, body:JSON.stringify(payload)});\n      const raw = await response.text(); let data;\n      try { data = JSON.parse(raw); } catch (_) { data = {detail:raw.slice(0,2500)}; }\n      data.http_status = response.status; show(data);\n    } catch (e) { show({detail:"Request failed", error:String(e)}); }\n    finally { button.disabled = false; }\n  });\n})();'

from pathlib import Path
import re
js_dir = APP_DIR / "static" / "ipl_api" / "js"
js_dir.mkdir(parents=True, exist_ok=True)
js_path = js_dir / "deep_learning_prediction_day32.js"
js_path.write_text(DL_JS_SOURCE, encoding="utf-8")
template_dir = APP_DIR / "templates"
changed = []
for p in template_dir.rglob("*.html"):
    old = p.read_text(encoding="utf-8", errors="ignore")
    if "Five-match sequence JSON" not in old and "DL sequence endpoint registered" not in old:
        continue
    new = old.replace('"preprocessed":true', '"preprocessed":false').replace('"preprocessed": true', '"preprocessed": false')
    if "data-day31-dl-submit" not in new:
        new = re.sub(r'<button\b([^>]*?)>\s*(?:Run DL prediction|DL endpoint not configured)\s*</button>',
                     lambda m: '<button data-day31-dl-submit="true"' + re.sub(r'\sdisabled(?:\s*=\s*(?:"disabled"|\'disabled\'|disabled))?', '', m.group(1), flags=re.I) + '>Run DL prediction</button>', new, flags=re.I)
    tag = '<script src="/static/ipl_api/js/deep_learning_prediction_day32.js"></script>'
    if "deep_learning_prediction_day32.js" not in new:
        new = new.replace("</body>", tag + "\n</body>") if "</body>" in new else new + "\n" + tag
    if new != old:
        p.write_text(new, encoding="utf-8"); changed.append(str(p.relative_to(ROOT)))
print("JS:", js_path.relative_to(ROOT))
print("Updated templates:", changed)


JS: django_api\ipl_api\static\ipl_api\js\deep_learning_prediction_day32.js
Updated templates: ['django_api\\ipl_api\\templates\\ipl_api\\dashboard_day25.html']


## Run syntax and Django checks

In [10]:
ENDPOINT_SOURCE = 'import json\nfrom pathlib import Path\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom django.conf import settings\nfrom django.http import JsonResponse\nfrom django.views.decorators.csrf import csrf_exempt\nfrom django.views.decorators.http import require_POST\n\nTARGETS = {"next_match_runs", "next_match_wickets", "next_match_runs_conceded"}\n\ndef repo_root():\n    base = Path(settings.BASE_DIR).resolve()\n    return base.parent if base.name == "django_api" else base\n\ndef resolve_artifact(root, value, base_dir=None):\n    p = Path(str(value))\n    candidates = [p] if p.is_absolute() else [root/p, root/"models"/p]\n    if base_dir and not p.is_absolute():\n        candidates.append(Path(base_dir)/p)\n    parts = list(p.parts)\n    if "models" in parts:\n        candidates.append(root.joinpath(*parts[parts.index("models"):]))\n    for c in candidates:\n        if c.exists():\n            return c.resolve()\n    return candidates[0].resolve()\n\ndef value_from(row, names):\n    for name in names:\n        value = row.get(name)\n        if value is not None and not pd.isna(value) and str(value).strip():\n            return str(value).strip()\n    return None\n\ndef find_manifest(root):\n    candidates = [root/"reports"/"day12"/"model_artifacts_manifest.csv",\n                  root/"reports"/"day12_model_artifacts_manifest.csv"]\n    if (root/"reports").exists():\n        candidates += list((root/"reports").rglob("*model*artifact*manifest*.csv"))\n    path = next((p for p in candidates if p.is_file()), None)\n    if path is None:\n        raise FileNotFoundError("Day 12 model_artifacts_manifest.csv not found under reports/.")\n    return pd.read_csv(path), path\n\n@csrf_exempt\n@require_POST\ndef deep_learning_prediction(request):\n    try:\n        payload = json.loads(request.body.decode("utf-8"))\n    except Exception:\n        return JsonResponse({"detail": "Request body must be valid JSON."}, status=400)\n    target = str(payload.get("target", "")).strip()\n    if target not in TARGETS:\n        return JsonResponse({"detail": "Unsupported target.", "allowed_targets": sorted(TARGETS)}, status=400)\n    if payload.get("preprocessed") is True:\n        return JsonResponse({"detail": "Send raw values in Day 11 feature order. The endpoint applies saved Day 12 preprocessing; remove preprocessed:true."}, status=400)\n    try:\n        sequence = np.asarray(payload.get("sequence"), dtype=np.float32)\n    except (TypeError, ValueError):\n        return JsonResponse({"detail": "sequence must contain numeric values."}, status=400)\n    if sequence.ndim != 2 or sequence.shape[0] != 5:\n        return JsonResponse({"detail": "sequence must be exactly 5 chronological rows.", "received_shape": list(sequence.shape)}, status=400)\n    if not np.isfinite(sequence).all():\n        return JsonResponse({"detail": "sequence contains NaN or infinite values."}, status=400)\n\n    root = repo_root()\n    try:\n        manifest, manifest_path = find_manifest(root)\n    except Exception as exc:\n        return JsonResponse({"detail": str(exc)}, status=503)\n    target_col = next((c for c in ["target", "target_name", "prediction_target"] if c in manifest.columns), None)\n    if not target_col:\n        return JsonResponse({"detail": "Manifest is missing a target column.", "columns": list(manifest.columns)}, status=503)\n    rows = manifest[manifest[target_col].astype(str).str.strip().eq(target)]\n    if rows.empty:\n        return JsonResponse({"detail": "No saved DL model is recorded for this target.", "target": target}, status=503)\n    row = rows.iloc[0].to_dict()\n    model_value = value_from(row, ["model_path", "model_file", "artifact_path"])\n    feature_value = value_from(row, ["feature_manifest_path", "features_path", "feature_path"])\n    if not model_value or not feature_value:\n        return JsonResponse({"detail": "Manifest row must include model_path and feature_manifest_path.", "columns": list(manifest.columns)}, status=503)\n    model_path = resolve_artifact(root, model_value, manifest_path.parent)\n    feature_path = resolve_artifact(root, feature_value, manifest_path.parent)\n    if not model_path.is_file():\n        return JsonResponse({"detail": "Saved DL model not found.", "path": str(model_path)}, status=503)\n    if not feature_path.is_file():\n        return JsonResponse({"detail": "Day 11 feature manifest not found.", "path": str(feature_path)}, status=503)\n    try:\n        feature_info = json.loads(feature_path.read_text(encoding="utf-8"))\n        features = feature_info.get("feature_columns_in_order", feature_info.get("feature_columns"))\n        if not isinstance(features, list) or not features:\n            raise ValueError("feature_columns_in_order missing or empty")\n    except Exception as exc:\n        return JsonResponse({"detail": "Cannot read ordered feature manifest.", "path": str(feature_path), "error": str(exc)}, status=503)\n    if sequence.shape[1] != len(features):\n        return JsonResponse({"detail": "Feature count mismatch.", "expected_feature_count": len(features),\n                             "received_feature_count": int(sequence.shape[1]), "feature_names_in_order": features,\n                             "hint": "Provide five rows with one number per feature in this exact order."}, status=400)\n\n    imputer_value = value_from(row, ["imputer_path", "imputer_file"])\n    scaler_value = value_from(row, ["scaler_path", "scaler_file"])\n    try:\n        flat = sequence.reshape(5, len(features))\n        if imputer_value:\n            path = resolve_artifact(root, imputer_value, manifest_path.parent)\n            if not path.is_file():\n                return JsonResponse({"detail": "Saved imputer not found.", "path": str(path)}, status=503)\n            flat = joblib.load(path).transform(flat)\n        if scaler_value:\n            path = resolve_artifact(root, scaler_value, manifest_path.parent)\n            if not path.is_file():\n                return JsonResponse({"detail": "Saved scaler not found.", "path": str(path)}, status=503)\n            flat = joblib.load(path).transform(flat)\n        x = np.asarray(flat, dtype=np.float32).reshape(1, 5, len(features))\n        import tensorflow as tf\n        model = tf.keras.models.load_model(model_path)\n        expected = tuple(model.input_shape[1:])\n        if len(expected) != 2 or expected[0] not in (None, 5) or expected[1] not in (None, len(features)):\n            return JsonResponse({"detail": "Model input shape does not match feature manifest.",\n                                 "model_input_shape": list(model.input_shape), "feature_count": len(features)}, status=503)\n        prediction = float(np.asarray(model.predict(x, verbose=0)).reshape(-1)[0])\n    except Exception as exc:\n        return JsonResponse({"detail": "DL inference failed while loading or applying saved artifacts.",\n                             "error": str(exc), "model_path": str(model_path)}, status=503)\n    return JsonResponse({"target": target, "prediction": prediction,\n                         "model": value_from(row, ["architecture", "model_type", "selected_architecture"]) or "Saved LSTM/GRU",\n                         "sequence_length": 5, "feature_count": len(features), "feature_names_in_order": features,\n                         "preprocessing": {"imputer_applied": bool(imputer_value), "scaler_applied": bool(scaler_value)}})\n'

import subprocess, sys
compile(ENDPOINT_SOURCE, "deep_learning_prediction_day32.py", "exec")
compile(URLS.read_text(encoding="utf-8"), str(URLS), "exec")
result = subprocess.run([sys.executable, "manage.py", "check"], cwd=DJANGO_DIR, capture_output=True, text=True)
print(result.stdout)
if result.stderr: print(result.stderr[-2500:])
print("manage.py check exit code:", result.returncode)
if result.returncode:
    raise RuntimeError("Django check failed; resolve the output before testing.")


System check identified no issues (0 silenced).

manage.py check exit code: 0


## Browser test checklist

1. Restart Django and hard-refresh `/api/dashboard/` (`Ctrl+F5`).
2. Enter a target and a sequence of **five chronological rows** with one numeric value per feature in the displayed Day 11 feature order.
3. Do not set `preprocessed:true`; the endpoint applies saved Day 12 imputer/scaler.
4. Test each target separately. If a target has no trained artifact, the API will report that instead of making up a prediction.
5. If the feature count is wrong, the response lists `expected_feature_count` and `feature_names_in_order`.
6. If a model/artifact is missing or incompatible, the response returns HTTP 503 with a diagnostic.

The old `[[0,0], ...]` sample is only valid if the model was trained with exactly two features; it should not be treated as real player data.